<a href="https://colab.research.google.com/github/Sayaniadak/demo-badge/blob/main/DS_TASK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import os
print(os.listdir("outputs/charts"))

['volume_by_sentiment.png', 'top_bottom_accounts.png', 'win_rate_by_sentiment.png', 'total_pnl_by_sentiment.png', 'pnl_vs_sentiment_timeseries.png']


In [39]:
# ---------------------------------------------------------------------------
# FILE PATHS - edit these if your filenames/locations differ
# ---------------------------------------------------------------------------
TRADER_FILE = "historical_data.csv"  # Assuming files will be uploaded to /content/
SENTIMENT_FILE = "fear_greed_index.csv" # Assuming files will be uploaded to /content/
OUTPUT_DIR = "outputs"
CHART_DIR = os.path.join(OUTPUT_DIR, "charts")
CHUNK_SIZE = 200_000  # rows per chunk; lower this if you hit memory errors

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHART_DIR, exist_ok=True)

sns.set_theme(style="darkgrid")

In [11]:
from google.colab import files
uploaded = files.upload()   # a button will appear — select both CSV files

Saving fear_greed_index.csv to fear_greed_index.csv


In [12]:
# ---------------------------------------------------------------------------
# STEP 1: Load sentiment data (small file, load directly)
# ---------------------------------------------------------------------------
print("Loading sentiment data...")
sentiment = pd.read_csv(SENTIMENT_FILE)
sentiment["date"] = pd.to_datetime(sentiment["date"]).dt.date
sentiment = sentiment[["date", "value", "classification"]].rename(
    columns={"value": "sentiment_value", "classification": "sentiment"}
)
print(f"  Sentiment rows: {len(sentiment)}")
print(f"  Sentiment classes: {sentiment['sentiment'].unique()}")

Loading sentiment data...
  Sentiment rows: 2644
  Sentiment classes: ['Fear' 'Extreme Fear' 'Neutral' 'Greed' 'Extreme Greed']


In [13]:
from google.colab import files
uploaded = files.upload()


Saving historical_data.csv to historical_data (1).csv


In [40]:
# Standardize expected column names - adjust left side if your headers differ
COLUMN_MAP = {
    "Account": "account",
    "Coin": "coin",
    "Execution Price": "execution_price",
    "Size Tokens": "size_tokens",
    "Size USD": "size_usd",
    "Side": "side",
    "Timestamp IST": "timestamp",
    "Start Position": "start_position",
    "Direction": "direction",
    "Closed PnL": "closed_pnl",
    "Transaction Hash": "tx_hash",
    "Order ID": "order_id",
    "Crossed": "crossed",
    "Fee": "fee",
    "Trade ID": "trade_id",
    "Leverage": "leverage",  # include if present in your file
}

daily_agg_list = []       # per (date, sentiment) rolled up per chunk
account_agg_list = []     # per (account) rolled up per chunk
sample_rows = []          # small sample for inspection

chunk_num = 0
for chunk in pd.read_csv(TRADER_FILE, chunksize=CHUNK_SIZE):
    chunk_num += 1
    print(f"  Chunk {chunk_num} ({len(chunk)} rows)...")

    # Rename only columns that exist
    rename_dict = {k: v for k, v in COLUMN_MAP.items() if k in chunk.columns}
    chunk = chunk.rename(columns=rename_dict)

    # Parse timestamp -> date
    chunk["timestamp"] = pd.to_datetime(chunk["timestamp"], errors="coerce", dayfirst=True)
    chunk["date"] = chunk["timestamp"].dt.date

    # Merge with sentiment
    chunk = chunk.merge(sentiment, on="date", how="left")

    # Keep a small sample from the first chunk only
    if chunk_num == 1:
        sample_rows.append(chunk.head(2000))

    # --- Daily aggregation (by date + sentiment) ---
    daily = chunk.groupby(["date", "sentiment", "sentiment_value"], dropna=False).agg(
        total_trades=("closed_pnl", "count"),
        total_closed_pnl=("closed_pnl", "sum"),
        avg_closed_pnl=("closed_pnl", "mean"),
        winning_trades=("closed_pnl", lambda x: (x > 0).sum()),
        losing_trades=("closed_pnl", lambda x: (x < 0).sum()),
        total_volume_usd=("size_usd", "sum"),
        avg_trade_size_usd=("size_usd", "mean"),
    ).reset_index()
    daily_agg_list.append(daily)

    # --- Account aggregation ---
    acct = chunk.groupby("account", dropna=False).agg(
        total_trades=("closed_pnl", "count"),
        total_closed_pnl=("closed_pnl", "sum"),
        winning_trades=("closed_pnl", lambda x: (x > 0).sum()),
        total_volume_usd=("size_usd", "sum"),
    ).reset_index()
    account_agg_list.append(acct)

print("\nAll chunks processed. Combining aggregates...")


  Chunk 1 (200000 rows)...
  Chunk 2 (11224 rows)...

All chunks processed. Combining aggregates...


In [42]:
# ---------------------------------------------------------------------------
# STEP 3: Combine chunk-level aggregates into final summaries
# ---------------------------------------------------------------------------
# Daily summary: re-aggregate across chunks (since same date can span chunks)
daily_all = pd.concat(daily_agg_list, ignore_index=True)
daily_summary = daily_all.groupby(["date", "sentiment", "sentiment_value"], dropna=False).agg(
    total_trades=("total_trades", "sum"),
    total_closed_pnl=("total_closed_pnl", "sum"),
    winning_trades=("winning_trades", "sum"),
    losing_trades=("losing_trades", "sum"),
    total_volume_usd=("total_volume_usd", "sum"),
).reset_index()
daily_summary["avg_pnl_per_trade"] = daily_summary["total_closed_pnl"] / daily_summary["total_trades"]
daily_summary["win_rate"] = daily_summary["winning_trades"] / daily_summary["total_trades"]
daily_summary = daily_summary.sort_values("date")
daily_summary.to_csv(os.path.join(OUTPUT_DIR, "daily_summary.csv"), index=False)
print(f"  Saved daily_summary.csv ({len(daily_summary)} rows)")

# Account summary: re-aggregate across chunks (same account can span chunks)
account_all = pd.concat(account_agg_list, ignore_index=True)
account_summary = account_all.groupby("account", dropna=False).agg(
    total_trades=("total_trades", "sum"),
    total_closed_pnl=("total_closed_pnl", "sum"),
    winning_trades=("winning_trades", "sum"),
    total_volume_usd=("total_volume_usd", "sum"),
).reset_index()
account_summary["win_rate"] = account_summary["winning_trades"] / account_summary["total_trades"]
account_summary = account_summary.sort_values("total_closed_pnl", ascending=False)
account_summary.to_csv(os.path.join(OUTPUT_DIR, "account_summary.csv"), index=False)
print(f"  Saved account_summary.csv ({len(account_summary)} rows)")

# Sample to inspect structure
sample_df = pd.concat(sample_rows, ignore_index=True)
sample_df.to_csv(os.path.join(OUTPUT_DIR, "cleaned_merged_sample.csv"), index=False)
print(f"  Saved cleaned_merged_sample.csv ({len(sample_df)} rows)")



  Saved daily_summary.csv (480 rows)
  Saved account_summary.csv (32 rows)
  Saved cleaned_merged_sample.csv (2000 rows)


In [23]:
# Sentiment-level summary (the key comparison table)
sentiment_summary = daily_summary.groupby("sentiment", dropna=False).agg(
    total_trades=("total_trades", "sum"),
    total_closed_pnl=("total_closed_pnl", "sum"),
    total_volume_usd=("total_volume_usd", "sum"),
    avg_daily_pnl=("total_closed_pnl", "mean"),
    avg_win_rate=("win_rate", "mean"),
    num_days=("date", "count"),
).reset_index().sort_values("total_closed_pnl", ascending=False)
sentiment_summary.to_csv(os.path.join(OUTPUT_DIR, "sentiment_summary.csv"), index=False)
print(f"  Saved sentiment_summary.csv")
print("\n", sentiment_summary)

# Sample for Claude to inspect structure
sample_df = pd.concat(sample_rows, ignore_index=True)
sample_df.to_csv(os.path.join(OUTPUT_DIR, "cleaned_merged_sample.csv"), index=False)
print(f"  Saved cleaned_merged_sample.csv ({len(sample_df)} rows)")


  Saved sentiment_summary.csv

        sentiment  total_trades  total_closed_pnl  total_volume_usd  \
2           Fear         61837      3.357155e+06      4.833248e+08   
1  Extreme Greed         39992      2.715171e+06      1.244652e+08   
3          Greed         50303      2.150129e+06      2.885825e+08   
4        Neutral         37686      1.292921e+06      1.802421e+08   
0   Extreme Fear         21400      7.391102e+05      1.144843e+08   
5            NaN             6      4.247199e+04      8.866886e+04   

   avg_daily_pnl  avg_win_rate  num_days  
2   36891.818040      0.329112        91  
1   23817.292199      0.467424       114  
3   11140.566181      0.335986       193  
4   19297.323516      0.331886        67  
0   52793.589178      0.327341        14  
5   42471.994130      1.000000         1  
  Saved cleaned_merged_sample.csv (2000 rows)


In [54]:
# ---------------------------------------------------------------------------
# STEP 4: Charts
# ---------------------------------------------------------------------------
print("\nGenerating charts...")

# Chart 1: Total PnL by sentiment
plt.figure(figsize=(9, 5))
sns.barplot(data=sentiment_summary, x="sentiment", y="total_closed_pnl", hue="sentiment", palette="viridis", legend=False)
plt.title("Total Closed PnL by Market Sentiment")
plt.ylabel("Total Closed PnL (USD)")
plt.xlabel("Sentiment")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "total_pnl_by_sentiment.png"), dpi=150)
plt.close()
plt.figure(figsize=(9,5))
sns.barplot(data=sentiment_summary, x="sentiment", y="total_closed_pnl", hue="sentiment", legend=False)
plt.title("Total Closed PnL by Sentiment")
plt.show()


Generating charts...


In [55]:
# Chart 2: Win rate by sentiment
plt.figure(figsize=(9, 5))
sns.barplot(data=sentiment_summary, x="sentiment", y="avg_win_rate", hue="sentiment", palette="magma", legend=False)
plt.title("Average Daily Win Rate by Market Sentiment")
plt.ylabel("Win Rate")
plt.xlabel("Sentiment")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "win_rate_by_sentiment.png"), dpi=150)
plt.close()
plt.figure(figsize=(9,5))
sns.barplot(data=sentiment_summary, x="sentiment", y="avg_win_rate", hue="sentiment", legend=False)
plt.title("Win Rate by Sentiment")
plt.show()

In [56]:
# Chart 3: Time series - sentiment value vs daily PnL
fig, ax1 = plt.subplots(figsize=(14, 6))
daily_summary_sorted = daily_summary.sort_values("date")
ax1.plot(daily_summary_sorted["date"], daily_summary_sorted["total_closed_pnl"],
          color="tab:blue", label="Daily Total Closed PnL")
ax1.set_ylabel("Total Closed PnL (USD)", color="tab:blue")
ax1.set_xlabel("Date")

ax2 = ax1.twinx()
ax2.plot(daily_summary_sorted["date"], daily_summary_sorted["sentiment_value"],
          color="tab:orange", alpha=0.5, label="Sentiment Index")
ax2.set_ylabel("Fear/Greed Index Value", color="tab:orange")

plt.title("Daily PnL vs Fear/Greed Index Over Time")
fig.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "pnl_vs_sentiment_timeseries.png"), dpi=150)
plt.close()
fig, ax1 = plt.subplots(figsize=(14,6))
ax1.plot(daily_summary["date"], daily_summary["total_closed_pnl"])
plt.title("PnL vs Sentiment Over Time")
plt.show()

In [57]:
# Chart 4: Trade volume by sentiment
plt.figure(figsize=(9, 5))
sns.barplot(data=sentiment_summary, x="sentiment", y="total_volume_usd", hue="sentiment", palette="crest", legend=False)
plt.title("Total Trading Volume (USD) by Market Sentiment")
plt.ylabel("Volume (USD)")
plt.xlabel("Sentiment")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "volume_by_sentiment.png"), dpi=150)
plt.close()
plt.figure(figsize=(9,5))
sns.barplot(data=sentiment_summary, x="sentiment", y="total_volume_usd", hue="sentiment", legend=False)
plt.title("Volume by Sentiment")
plt.show()

In [58]:
# Chart 5: Top 20 and bottom 20 accounts by PnL
top20 = account_summary.head(20)
bottom20 = account_summary.tail(20)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=top20, y="account", x="total_closed_pnl", hue="account", ax=axes[0], palette="Greens_r", legend=False)
axes[0].set_title("Top 20 Accounts by Total Closed PnL")
sns.barplot(data=bottom20, y="account", x="total_closed_pnl", hue="account", ax=axes[1], palette="Reds_r", legend=False)
axes[1].set_title("Bottom 20 Accounts by Total Closed PnL")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "top_bottom_accounts.png"), dpi=150)
plt.close()
fig, axes = plt.subplots(1, 2, figsize=(16,6))
sns.barplot(data=top20, y="account", x="total_closed_pnl", ax=axes[0])
sns.barplot(data=bottom20, y="account", x="total_closed_pnl", ax=axes[1])
plt.show()

print(f"\nDone! All outputs saved in: {os.path.abspath(OUTPUT_DIR)}")
print("Upload the CSV files (not charts, not the raw historical_data.csv) ")



Done! All outputs saved in: /content/outputs
Upload the CSV files (not charts, not the raw historical_data.csv) 
